# 第6章 断面交通量多提前量预测

本工作本是一份连续的实践，不另分学生任务版与参考版。按单元逐步运行，在参数区修改条件，记录自己的结果和解释；不要只执行整章脚本后截一张图。

**协议：** `ch06-forecast-v1`  
**必做：** 历史星期小时均值；上周同期；ARIMA(2,0,0)；SARIMA(2,0,0)(1,0,0)24  
**对象：** 西向单断面小时交通量：辆/小时  
**样本：** 2018年7-9月，1/3/6小时共同2169个目标时刻  
**划分：** 2017训练（8760标签、47缺测）；2018上半年验证；7-9月公开测试  
**比较：** 所有模型、所有提前量用相同2169目标；只取起点t-h的过滤状态；NaN不补0

先解压完整资料包，在其目录内运行。安装 `python -m pip install -r chapters/python/learning-requirements.txt`，再启动Jupyter。代码只读取固定真实数据；输出写入`outputs/chXX/`，不覆盖原数据。

每一步的“请解释”需用本次实际结果回答。默认参考配置可直接运行，但自动生成文件不代表学生已完成分析。


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'chapters/data').is_dir() and (p/'projects/data').is_dir())
sys.path.insert(0, str(ROOT/'chapters/python'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
from learning_support import metrics, metric_table, csv_file, primary, predictions_table, report, save_plot
THREADS = threadpool_limits(limits=1)
print('Data root:', ROOT)


## 1. 建立真实时间骨架和共同目标
请解释：主实验采用2169个共同目标，与旧1小时2190目标有什么区别？不要把缺失小时压缩成相邻小时。


In [ ]:
FILES=['projects/data/audit.json','projects/data/forecast.json']
audit=json.loads((ROOT/FILES[0]).read_text(encoding='utf-8'))['rows']
frozen=json.loads((ROOT/FILES[1]).read_text(encoding='utf-8'))['horizons']
index=pd.date_range('2017-01-01','2018-09-30T23:00',freq='h')
source=pd.Series({pd.Timestamp(r[0]):r[1] for r in audit},dtype=float)
series=source.reindex(index)/1000
train=series.loc[:'2017-12-31T23:00']
common=sorted(set.intersection(*[set(r[0] for r in frozen[str(h)]['rows']) for h in [1,3,6]]))
targets=index.get_indexer(pd.to_datetime(common))
assert len(common)==2169 and len(train)==8760 and train.isna().sum()==47
actual=series.iloc[targets].to_numpy()*1000


## 2. 在训练期实际拟合模型
先运行下面预定阶数。若比较其他阶数，只根据验证期评价选择，再冻结参数；不得挑选测试期表现最好的一组当作独立结果。


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from case_algorithms import state_forecasts
DEFINITIONS=[('ARIMA',(2,0,0),(0,0,0,0)),('SARIMA',(2,0,0),(1,0,0,24))]
fitted_models={}
for name,order,seasonal in DEFINITIONS:
    model=SARIMAX(train,order=order,seasonal_order=seasonal,trend='c').fit(disp=False,maxiter=150)
    assert model.mle_retvals['converged'], name+' did not converge'
    fitted_models[name]=model
    print(name,model.params.to_dict())


## 3. 检查验证期与训练残差
极小p值提示仍有自相关，不等于模型完全无效。数值下溢为0不表示数学上的概率恰为零。


In [ ]:
val_positions=np.flatnonzero((index>='2018-01-01')&(index<'2018-07-01')&series.notna())
filtered_models={};diagnostics=[]
for name,order,seasonal in DEFINITIONS:
    fitted=fitted_models[name]
    filtered=SARIMAX(series,order=order,seasonal_order=seasonal,trend='c').filter(fitted.params)
    filtered_models[name]=filtered
    validation_prediction=state_forecasts(filtered,val_positions,[1])[1]*1000
    residual=pd.Series(fitted.filter_results.standardized_forecasts_error[0],index=train.index).where(train.notna()).iloc[72:]
    longest=max((g.dropna() for _,g in residual.groupby(residual.isna().cumsum())),key=len)
    pvalue=float(acorr_ljungbox(longest,lags=[24],model_df=3 if seasonal[0] else 2).lb_pvalue.iloc[0])
    diagnostics.append({'model':name,'validation_RMSE':metrics(series.iloc[val_positions].to_numpy()*1000,validation_prediction)['RMSE'],'residual_n':len(longest),'LjungBox24_p':pvalue})
display(pd.DataFrame(diagnostics))


## 4. 按起点过滤状态滚动预测
函数只取t-h的过滤状态，不使用平滑状态。阅读`case_algorithms.py`的`state_forecasts`，说明为什么修改未来观测不应改变同一起点的预测。


In [ ]:
predictions={}
for name,model in filtered_models.items():
    for h,values in state_forecasts(model,targets,[1,3,6]).items():
        predictions[f'{name}|h={h}']=values*1000
for h in [1,3,6]:
    rows={r[0]:r for r in frozen[str(h)]['rows']}
    predictions[f'Week|h={h}']=np.array([rows[t][4] for t in common])
    predictions[f'Calendar|h={h}']=np.array([rows[t][5] for t in common])
scores=metric_table(actual,predictions);display(scores)
peak_mask=np.isin(pd.to_datetime(common).hour,[7,8,9,16,17,18])
peak=metric_table(actual[peak_mask],{k:v[peak_mask] for k,v in predictions.items()});display(peak)


## 5. 预先固定图示窗口与失败样本
图只展示前72个共同目标，不挑最好片段。失败样本按绝对误差排序用于诊断，不能事后删掉再宣称改善。


In [ ]:
HORIZON=3  # 可改为1或6
plt.figure(figsize=(11,4));plt.plot(common[:72],actual[:72],label='Observed')
for name in ['ARIMA','SARIMA','Calendar']:
    plt.plot(common[:72],predictions[f'{name}|h={HORIZON}'][:72],label=name)
plt.xticks([0,24,48,71],[common[i][5:16] for i in [0,24,48,71]],rotation=20)
plt.ylabel('Vehicles/hour');plt.legend();save_plot(ROOT,6,'forecast_window')
worst=np.argsort(-abs(predictions[f'SARIMA|h={HORIZON}']-actual))[:10]
failures=pd.DataFrame({'target':np.array(common)[worst],'actual':actual[worst],'prediction':predictions[f'SARIMA|h={HORIZON}'][worst]})
display(failures)


## 6. 交付主实验结果与运行适用性报告
JSON包含四方法三个提前量，不混入旧2190行CSV接口。


In [ ]:
outputs={'predictions':predictions_table(common,predictions),'diagnostics':diagnostics,'peak':peak.to_dict('records')}
primary(ROOT,6,FILES,{'definitions':DEFINITIONS,'horizons':[1,3,6],'fit_end':'2017-12-31T23:00'},outputs)
csv_file(ROOT/'outputs/ch06/predictions.csv',outputs['predictions']['columns'],outputs['predictions']['rows'])
csv_file(ROOT/'outputs/ch06/failures.csv',failures.columns,failures.values)
report(ROOT,6,'断面交通量预测适用性报告',{'共同目标':len(common),'全期误差':scores.to_string(index=False),'高峰误差':peak.to_string(index=False),'失败样本':failures.to_string(index=False)},['不同提前量分别适合什么运行任务？','MAE与RMSE的排序为什么不同？','用于拥堵预警还缺哪些变量与验证？'])
